In [ ]:
!pip install chromadb pypdf fastapi uvicorn

In [ ]:
# ingest.py
import os   
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

load_dotenv()

DOCS_DIR = "docs"
DB_DIR = "chroma_db"

def load_pdfs(folder: str):
    docs = []
    for fname in os.listdir(folder):
        if not fname.lower().endswith(".pdf"):
            continue
        path = os.path.join(folder, fname)
        loader = PyPDFLoader(path)
        file_docs = loader.load()
        # Add source metadata (filename) if not present
        for d in file_docs:
            d.metadata.setdefault("source", fname)
        docs.extend(file_docs)
    return docs

def main():
    print("Loading PDFs...")
    docs = load_pdfs(DOCS_DIR)
    print(f"Loaded {len(docs)} pages from PDFs")

    print("Splitting into chunks...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    print(f"Created {len(chunks)} chunks")

    print("Creating embeddings with Gemini...")
    embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

    print("Building Chroma vector store...")
    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_DIR,
    )
    vectordb.persist()
    print(f"Vector store saved to {DB_DIR}")

if __name__ == "__main__":
    main()
